# Demo del orquestador multi-agente

Este notebook demuestra el **flujo de delegación** de la pre-entrega 6: quién decide, a quién
le pasa la tarea, con qué instrucción y con qué aporte vuelve.

Corre **dos consultas** para mostrar que el equipo se arma según el pedido:

1. Una que necesita **los dos dominios**: el dato (cuántos días de vacaciones por tramo de
   antigüedad) lo tiene el **investigador** en las políticas internas, y la cuenta total la
   hace el **analista** con la calculadora. El analista no puede correr antes que el
   investigador: los números que usa son los que el investigador trajo.
2. Una que se responde con **un solo dominio** (no hay cuenta que hacer): el supervisor
   cierra con el investigador y no llama al analista al pedo.

El orden lo fija el **supervisor**; los especialistas no se conocen entre sí.


## 1. La topología del grafo

Se arma una sola vez y se inspecciona. Los especialistas siempre vuelven al supervisor (ese
es el ciclo), y el nodo de `validacion` es el que decide si ya se puede cerrar.


In [1]:
from graph import build_app, diagrama_mermaid

app = build_app()
print(diagrama_mermaid(app))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	investigador(investigador)
	analista(analista)
	validacion(validacion)
	sintesis(sintesis)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analista --> supervisor;
	investigador --> supervisor;
	supervisor -.-> analista;
	supervisor -.-> investigador;
	supervisor -.-> validacion;
	validacion -.-> sintesis;
	validacion -.-> supervisor;
	sintesis --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. El dataset y el recuperador

El investigador no inventa: busca en el mismo dataset de las pre-entregas anteriores
(políticas internas de una empresa ficticia). Los documentos son largos como para partirse en
varios fragmentos y el índice de Pinecone se puebla desde `data/` con `ingest.py`, así que el
índice y los archivos no se separan nunca.

Acá se ve también **en qué modo quedó el recuperador**: híbrido (BM25 + vectorial) si hay
Pinecone, léxico si no. Si falta la clave, se avisa: no se degrada en silencio.


In [2]:
import tiktoken
from rag import RAGSystem, chunk_documents, load_dataset

codificador = tiktoken.get_encoding('cl100k_base')
documentos = load_dataset()
print('DATASET')
print('=' * 70)
for documento in documentos:
    tokens = len(codificador.encode(documento.page_content))
    print(f"  {documento.metadata['source']:45s} {tokens:5d} tokens")
print(f"  total: {len(documentos)} documentos, "
      f"{sum(len(codificador.encode(d.page_content)) for d in documentos)} tokens")

chunks = chunk_documents(documentos)
print(f"\n  fragmentos (chunks de 600 tokens): {len(chunks)}")

sistema = RAGSystem()
print(f"\n  recuperador: {sistema.modo}")

DATASET
  onboarding_nuevos_empleados.txt                 972 tokens
  politica_seguridad_informatica.txt             1093 tokens
  politica_vacaciones.txt                        1634 tokens
  politica_teletrabajo.txt                       1357 tokens
  total: 4 documentos, 5056 tokens



  fragmentos (chunks de 600 tokens): 15



  recuperador: híbrido (BM25 + vectorial sobre Pinecone)


## 3. La consulta que necesita los dos dominios

`correr()` recorre el grafo y va mostrando cada paso: la decisión del supervisor (con la
instrucción que le deja al especialista), el aporte que devuelve cada agente y el informe del
validador. Al final imprime la respuesta para el usuario.


In [3]:
from main import correr

resultado, traza = correr()

PREGUNTA: Un colaborador cumplió 12 años en la empresa: 2 años con menos de 3 de antigüedad, 6 años con entre 3 y 8, y 4 años con más de 8. ¿Cuántos días de vacaciones le correspondieron en total a lo largo de esos 12 años?

FLUJO DE DELEGACIÓN
----------------------------------------------------------------------------------------


🔹 supervisor    decide -> investigador | instrucción: Buscá la política interna de vacaciones de la empresa que especifique los días correspondi


🔹 investigador  [investigador] [Fuente: politica_vacaciones.txt]
- Menos de 3 años de antigüedad: 15 días hábiles de vacaciones por año calendario.
- 3 años o más, y hasta 8 años de antigüedad: 20 días hábiles por año calendario.
- Más de 8 años de an


🔹 supervisor    decide -> analista | instrucción: Calculá con los números que ya están en el contexto la cuenta que responde la pregunta ori


🔹 analista      [analista] Resultado: 250
Cuenta realizada: 2*15 + 6*20 + 4*25


🔹 supervisor    decide -> FINISH
🔹 validacion    suficiente=True | Los dominios exigidos (investigador, analista) aportaron de forma verificable y ninguno devolvió error: se puede sintetizar.


🔹 sintesis      [supervisor] A lo largo de los 12 años, al colaborador le correspondieron en total **250 días hábiles de vacaciones**.

El cálculo se desglosa de la siguiente manera según la política de la empresa:

* **2 años con menos de 3 de anti

----------------------------------------------------------------------------------------
RESPUESTA FINAL

A lo largo de los 12 años, al colaborador le correspondieron en total **250 días hábiles de vacaciones**.

El cálculo se desglosa de la siguiente manera según la política de la empresa:

* **2 años con menos de 3 de antigüedad:** 2 × 15 días = 30 días hábiles.
* **6 años con entre 3 y 8 de antigüedad:** 6 × 20 días = 120 días hábiles.
* **4 años con más de 8 de antigüedad:** 4 × 25 días = 100 días hábiles.

**Total:** 30 + 120 + 100 = **250 días hábiles**.
----------------------------------------------------------------------------------------
pasos de especialistas: 2 | contribuciones: 2 | rúbrica cumplida: True
traza guardada en traz

## 4. El estado compartido: quién aportó qué

Esto es lo que evita la pérdida de contexto. Cada contribución queda registrada con su autor
(el reducer `operator.add` las acumula en vez de pisarlas), y el supervisor decide mirando esa
lista.


In [4]:
print('RESPUESTA FINAL:\n')
print(traza['respuesta_final'])

print('\n' + '=' * 70)
print('CONTRIBUCIONES (quién aportó qué)')
print('=' * 70)
for c in traza['contribuciones']:
    print(f"\n[{c['agente']}]\n{c['aporte']}")

print('\n' + '=' * 70)
print('INFORME DEL VALIDADOR')
print('=' * 70)
print(traza['validacion'])
print(f"\nPasos de especialistas: {traza['pasos_ejecutados']}")

RESPUESTA FINAL:

A lo largo de los 12 años, al colaborador le correspondieron en total **250 días hábiles de vacaciones**.

El cálculo se desglosa de la siguiente manera según la política de la empresa:

* **2 años con menos de 3 de antigüedad:** 2 × 15 días = 30 días hábiles.
* **6 años con entre 3 y 8 de antigüedad:** 6 × 20 días = 120 días hábiles.
* **4 años con más de 8 de antigüedad:** 4 × 25 días = 100 días hábiles.

**Total:** 30 + 120 + 100 = **250 días hábiles**.

CONTRIBUCIONES (quién aportó qué)

[investigador]
[Fuente: politica_vacaciones.txt]
- Menos de 3 años de antigüedad: 15 días hábiles de vacaciones por año calendario.
- 3 años o más, y hasta 8 años de antigüedad: 20 días hábiles por año calendario.
- Más de 8 años de antigüedad: 25 días hábiles por año calendario.

[analista]
Resultado: 250
Cuenta realizada: 2*15 + 6*20 + 4*25

INFORME DEL VALIDADOR
{'suficiente': True, 'faltantes': [], 'dominios_con_aporte': ['investigador', 'analista'], 'dominios_exigidos': ['inv

## 5. La traza, paso a paso

La misma corrida, leída del `traza_ejecucion.json` que queda en el repo: el orden en que
trabajó cada nodo, con la decisión y la instrucción de cada paso.


In [5]:
for i, paso in enumerate(traza['pasos'], 1):
    print(f"{i}. {paso['nodo']:13s} {paso['detalle']}")

1. supervisor    decide -> investigador | instrucción: Buscá la política interna de vacaciones de la empresa que especifique los días correspondi
2. investigador  [investigador] [Fuente: politica_vacaciones.txt]
- Menos de 3 años de antigüedad: 15 días hábiles de vacaciones por año calendario.
- 3 años o más, y hasta 8 años de antigüedad: 20 días hábiles por año calendario.
- Más de 8 años de an
3. supervisor    decide -> analista | instrucción: Calculá con los números que ya están en el contexto la cuenta que responde la pregunta ori
4. analista      [analista] Resultado: 250
Cuenta realizada: 2*15 + 6*20 + 4*25
5. supervisor    decide -> FINISH
6. validacion    suficiente=True | Los dominios exigidos (investigador, analista) aportaron de forma verificable y ninguno devolvió error: se puede sintetizar.
7. sintesis      [supervisor] A lo largo de los 12 años, al colaborador le correspondieron en total **250 días hábiles de vacaciones**.

El cálculo se desglosa de la siguiente manera se

## 6. La consulta que NO necesita cuenta

Esta pregunta se responde con el dato de la política. El punto es ver dos cosas:

- que la **rúbrica se adapta**: el validador exige el dominio del análisis sólo cuando la
  pregunta pide una cuenta (si no, el flujo nunca cerraría);
- que el supervisor **vuelve a delegar** al investigador con otra instrucción cuando falta la
  segunda parte de la pregunta, y recién después cierra. Son iteraciones reales del ciclo, no
  una sola vuelta.


In [6]:
from main import PREGUNTA_SIMPLE

resultado_simple, traza_simple = correr(PREGUNTA_SIMPLE, archivo_traza='traza_consulta_simple.json')

print('\n' + '=' * 70)
print('RÚBRICA DE ESTA CONSULTA')
print('=' * 70)
informe = traza_simple['validacion']
print('dominios exigidos :', informe['dominios_exigidos'])
print('dominios que aportaron:', informe['dominios_con_aporte'])
print('suficiente:', informe['suficiente'])

print('\nPASOS:')
for i, paso in enumerate(traza_simple['pasos'], 1):
    print(f"{i}. {paso['nodo']:13s} {paso['detalle']}")

print('\nRESPUESTA FINAL:\n')
print(traza_simple['respuesta_final'])

PREGUNTA: ¿Se puede tomar vacaciones en enero y qué pasa con los días que quedan sin usar al cerrar el año?

FLUJO DE DELEGACIÓN
----------------------------------------------------------------------------------------


🔹 supervisor    decide -> investigador | instrucción: Buscá en la política interna de vacaciones si está permitido tomarlas en el mes de enero y


🔹 investigador  [investigador] **Vacaciones en enero:**
«Durante el mes de enero se aplica un régimen especial de guardias mínimas para garantizar la continuidad de los proyectos de clientes, por lo que las solicitudes de vacaciones en ese período est


🔹 supervisor    decide -> FINISH
🔹 validacion    suficiente=True | Los dominios exigidos (investigador) aportaron de forma verificable y ninguno devolvió error: se puede sintetizar.


🔹 sintesis      [supervisor] Respecto a tus consultas:

* **Vacaciones en enero:** Sí, podés tomarlas, pero están sujetas a un régimen de guardias mínimas con un cupo máximo de hasta el **25% del equipo** en simultáneo. Además, la solicitud debe pre

----------------------------------------------------------------------------------------
RESPUESTA FINAL

Respecto a tus consultas:

* **Vacaciones en enero:** Sí, podés tomarlas, pero están sujetas a un régimen de guardias mínimas con un cupo máximo de hasta el **25% del equipo** en simultáneo. Además, la solicitud debe presentarse **antes del 30 de noviembre** del año anterior.
* **Días no utilizados al cierre del año:** Se pueden acumular como máximo **hasta 5 días** para el siguiente período; el resto de los días que superen ese tope se pierden. La única excepción es contar con una justificación de negocio documentada por tu líder y aprobada por Personas antes del 31 de diciembre (o autorización expresa por razones operativas).
-------

## 7. Qué demuestra esto

- **Topología jerárquica**: un supervisor que rutea y decide el final; dos especialistas con
  herramientas acotadas (búsqueda y cálculo) que no se llaman entre ellos.
- **Estado compartido estructurado**: `contribuciones` con reducer `operator.add`, más
  `next_agent`, `instruccion`, `pasos`, `task_completed` y el informe de `validacion`.
- **Flujo de supervisión con refinamiento**: el supervisor delega, recibe el aporte, decide si
  falta algo y puede volver a delegar con otra instrucción.
- **Corte del "Supervisor Infinito"**: `MAX_PASOS` en el supervisor, el validador determinista
  y `MAX_REINTENTOS` en las aristas.
- **Contexto acotado**: a cada especialista le llega una instrucción puntual, la pregunta
  original y los aportes (no el historial entero).
- **Errores que no rompen el flujo**: si el proveedor rechaza una llamada, el especialista
  devuelve un aporte marcado como `ERROR:` con el motivo, y el equipo decide qué hacer.
